In [ ]:
from google.colab import userdata
import requests
import time
import datetime
import pandas as pd

api_key = userdata.get("RIOT_API_KEY")

headers = {"X-Riot-Token": api_key}

# get ur PUUID from riot API: replace with your user and game tag
game_name = "jackdzn"
tag_line = "NA1"

url = f"https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}"
response = requests.get(url, headers=headers)
data = response.json()

#Check to see if issue grabbing puuid
puuid = data["puuid"]
print("PUUID loaded:", "Yes" if puuid else "Not found")
time.sleep(1.2)

# Pull games: Riot API only allows you to pull 100 games at a time - so here I try to pull 2 x 100 games
url_all = f"https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?count=100&start=0"
url_all_2 = f"https://americas.api.riotgames.com/lol/match/v5/matches/by-puuid/{puuid}/ids?count=100&start=100"

batch_1 = requests.get(url_all,   headers=headers).json()
time.sleep(1.2)
batch_2 = requests.get(url_all_2, headers=headers).json()
time.sleep(1.2)

matches = list(set(batch_1 + batch_2))
print(f"Batch 1: {len(batch_1)} | Batch 2: {len(batch_2)}")
print(f"Total unique matches: {len(matches)}\n")

#loop every game to find metadata
all_games = []

for i, match_id in enumerate(matches):
    try:
        url = f"https://americas.api.riotgames.com/lol/match/v5/matches/{match_id}"
        response = requests.get(url, headers=headers)
        match_data = response.json()
        time.sleep(1.2)

        #Find your own entry - since games record all 10 players
        my_stats = None
        for player in match_data['info']['participants']:
            if player['riotIdGameName'].lower() == 'jackdzn': #CHANGE to your own user
                my_stats = player
                break

        if my_stats is None:
            print(f"[{i+1}/{len(matches)}] Skipped — jackdzn not found in {match_id}")
            continue

        #Game Metadata
        game_mode = match_data['info']['gameMode']
        queue_id = match_data['info'].get('queueId', 0)
        duration_mins = match_data['info']['gameDuration'] / 60
        timestamp = match_data['info']['gameStartTimestamp'] / 1000
        game_time = datetime.datetime.fromtimestamp(timestamp)

        #If you dont care abbout certain game modes: Skip here: ex. Arena
        if game_mode == 'CHERRY':
            print(f"[{i+1}/{len(matches)}] Skipped Arena — {match_id}")
            continue

        # Readable mode label
        if game_mode == 'ARAM':
            mode_label = 'ARAM'
        elif game_mode in ('CLASSIC', 'SWIFTPLAY') and queue_id == 490:
            mode_label = 'Swiftplay'
        elif game_mode in ('CLASSIC', 'SWIFTPLAY') and queue_id == 400:
            mode_label = 'Draft'
        elif game_mode in ('CLASSIC', 'SWIFTPLAY') and queue_id == 430:
            mode_label = 'Blind'
        elif game_mode in ('CLASSIC', 'SWIFTPLAY'):
            mode_label = 'SR_Other'
        else:
            mode_label = game_mode

        #Find specific SR data
        is_sr = game_mode in ('CLASSIC', 'SWIFTPLAY')

        # Build stat row - Claude helped streamline this part:
        row = {
            #Identity
            'match_id'           : match_id,
            'champion'           : my_stats['championName'],
            'game_mode'          : mode_label,
            'win'                : my_stats['win'],

            #Ttime
            'game_date'          : game_time.strftime('%Y-%m-%d'),
            'hour_of_day'        : game_time.hour,
            'day_of_week'        : game_time.strftime('%A'),
            'duration_mins'      : round(duration_mins, 1),

            #Core stats
            'kills'              : my_stats['kills'],
            'deaths'             : my_stats['deaths'],
            'assists'            : my_stats['assists'],
            'kda'                : my_stats['challenges'].get('kda', 0),
            'kill_participation' : round(my_stats['challenges'].get('killParticipation', 0) * 100, 1),
            'damage_per_min'     : round(my_stats['challenges'].get('damagePerMinute', 0), 1),
            'team_damage_pct'    : round(my_stats['challenges'].get('teamDamagePercentage', 0) * 100, 1),
            'gold_per_min'       : round(my_stats['challenges'].get('goldPerMinute', 0), 1),
            'total_damage'       : my_stats['totalDamageDealtToChampions'],
            'total_damage_taken' : my_stats['totalDamageTaken'],
            'damage_mitigated'   : my_stats['damageSelfMitigated'],

            #Mechanics
            'skillshots_hit'     : my_stats['challenges'].get('skillshotsHit', 0),
            'skillshots_dodged'  : my_stats['challenges'].get('skillshotsDodged', 0),
            'dodge_small_window' : my_stats['challenges'].get('dodgeSkillShotsSmallWindow', 0),
            'immobilizations'    : my_stats['challenges'].get('enemyChampionImmobilizations', 0),
            'solo_kills'         : my_stats['challenges'].get('soloKills', 0),
            'ability_uses'       : my_stats['challenges'].get('abilityUses', 0),

            #Survivability
            'champ_level'        : my_stats['champLevel'],
            'time_spent_dead'    : my_stats['totalTimeSpentDead'],
            'longest_life'       : my_stats['longestTimeSpentLiving'],
            'largest_spree'      : my_stats['largestKillingSpree'],
            'largest_multikill'  : my_stats['largestMultiKill'],

            #SR + Swiftplay only (None if ARAM)
            'cs'                 : my_stats['totalMinionsKilled'] if is_sr else None,
            'cs_per_min'         : round(my_stats['totalMinionsKilled'] / duration_mins, 2) if is_sr else None,
            'vision_score'       : my_stats['visionScore'] if is_sr else None,
            'vision_per_min'     : round(my_stats['challenges'].get('visionScorePerMinute', 0), 2) if is_sr else None,
            'lane'               : my_stats.get('lane') if is_sr else None,
            'role'               : my_stats.get('role') if is_sr else None,
        }

        all_games.append(row)
        print(f"[{i+1}/{len(matches)}] ✓ {row['champion']} | {mode_label} | {'WIN' if row['win'] else 'LOSS'} | KDA: {row['kda']}")

    except Exception as e:
        print(f"[{i+1}/{len(matches)}] Error on {match_id}: {e}")
        continue

#Save data into a CSV so you can inport to Excel for cleaning/analysis
df = pd.DataFrame(all_games)
df.to_csv('jackdzn_match_history.csv', index=False) #CHANGE to your own user

print(f"\n✅ Done! {len(all_games)} games saved to jackdzn_match_history.csv")
print(f"\nPreview:")
print(df.head())
print(f"\nGame mode breakdown:")
print(df['game_mode'].value_counts())
print(f"\nWin rate: {round(df['win'].mean() * 100, 1)}%")
print(f"Average KDA: {round(df['kda'].mean(), 2)}")
print(f"Average CS/min (SR only): {round(df[df['cs_per_min'].notna()]['cs_per_min'].mean(), 2)}")
print(f"Average vision/min (SR only): {round(df[df['vision_per_min'].notna()]['vision_per_min'].mean(), 2)}")
print(f"\nMost played champions:\n{df['champion'].value_counts().head(5)}")

In [ ]:
from google.colab import files
#Download CSV - When you upload to Excel and want to summarize/make changes dont forget to save as workbook
files.download('match_history.csv')